# Evaluation of sequence-length effects on amino-acid-composition PCA

This notebook evaluates sequence-length effects separately for the DBD, full-length (FL), and non-DBD datasets.

For each dataset it:

1. reproduces the primary PCA using the descriptor-correlation threshold `|r| > 0.60`;
2. calculates Spearman correlations between `log10(sequence length)` and PC scores;
3. applies Benjamini-Hochberg correction across the tested dataset-PC combinations;
4. removes the linear effect of `log10(sequence length)` from every retained standardized descriptor and repeats PCA using the residuals;
5. compares the original and length-adjusted PC1-PC2 configurations using pairwise-distance Spearman correlation and Procrustes disparity.

Exact sequence lengths cannot be reconstructed from MolePct values. Upload either a FASTA file for each dataset or CSV files that already contain a recognized sequence-length column.


In [ ]:
# Upload the three MolePct CSV files and, normally, the three corresponding FASTA files.
from google.colab import files

print('Upload all input CSV and FASTA files together.')
uploaded = files.upload()
print('Uploaded files:', sorted(uploaded))


In [ ]:
from pathlib import Path
import re
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.spatial.distance import pdist
from scipy.spatial import procrustes
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

BASE_DIR = Path('/content')
OUTPUT_DIR = BASE_DIR / 'PCA_sequence_length_effects'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CORRELATION_THRESHOLD = 0.60
N_COMPONENTS = 10
PCS_TO_TEST = 10

# Change only the FASTA filename values if your FASTA files have different names.
DATASETS = {
    'DBD': {
        'csv': BASE_DIR / 'aaComp_MolePct.csv',
        'fasta': BASE_DIR / 'aor0-5_SM00066_domains_2.faa',
        'id_mode': 'SM00066',
    },
    'FL': {
        'csv': BASE_DIR / 'FL_aaComp_MolePct.csv',
        'fasta': BASE_DIR / 'aor0-5_SM00066_FL.faa',
        'id_mode': 'FL',
    },
    'non_DBD': {
        'csv': BASE_DIR / 'non_DBD_aaComp_MolePct.csv',
        'fasta': BASE_DIR / 'aor0-5_non_SM00066_domains_AO090012000182_fixed.faa',
        'id_mode': 'non_DBD',
    },
}

# Recognized length columns. If one exists in a CSV, its values are used and FASTA is unnecessary.
LENGTH_COLUMN_CANDIDATES = [
    'Length', 'length', 'SequenceLength', 'sequence_length',
    'SeqLength', 'seq_length', 'Length_aa', 'length_aa',
]


In [ ]:
def clean_ids(values, mode):
    """Apply the same ID convention as the primary PCA notebook."""
    base = pd.Series(values, dtype='object').astype(str)
    base = base.str.split('|', n=1, expand=True)[0]
    base = base.str.replace('-P', '', regex=False)

    if mode == 'SM00066':
        counts = base.groupby(base).transform('count')
        occurrence = base.groupby(base).cumcount() + 1
        base = pd.Series(
            np.where(counts > 1, base + '_' + occurrence.astype(str), base),
            index=base.index,
        )
    elif mode not in {'FL', 'non_DBD'}:
        raise ValueError(f'Unknown ID mode: {mode}')
    return base.astype(str).tolist()


def read_fasta_lengths(path, mode):
    """Read FASTA without requiring Biopython and return cleaned ID -> amino-acid length."""
    headers, sequences = [], []
    current_header, current_sequence = None, []

    with open(path, encoding='utf-8') as handle:
        for raw in handle:
            line = raw.strip()
            if not line:
                continue
            if line.startswith('>'):
                if current_header is not None:
                    headers.append(current_header)
                    sequences.append(''.join(current_sequence))
                current_header = line[1:].split()[0]
                current_sequence = []
            else:
                current_sequence.append(line.replace(' ', ''))
        if current_header is not None:
            headers.append(current_header)
            sequences.append(''.join(current_sequence))

    if not headers:
        raise ValueError(f'No FASTA records found: {path}')

    cleaned = clean_ids(headers, mode)
    table = pd.DataFrame({
        'ID': cleaned,
        'sequence_length_aa': [len(seq.rstrip('*')) for seq in sequences],
    })
    duplicated = table.loc[table['ID'].duplicated(keep=False), 'ID'].unique().tolist()
    if duplicated:
        raise ValueError(f'Duplicate cleaned FASTA IDs in {path}: {duplicated[:10]}')
    return table


def get_lengths(df, clean_sample_ids, fasta_path, id_mode):
    """Use a CSV length column when present; otherwise match lengths from FASTA."""
    length_column = next((c for c in LENGTH_COLUMN_CANDIDATES if c in df.columns), None)
    if length_column is not None:
        length_values = pd.to_numeric(df[length_column], errors='coerce')
        source = f'CSV column: {length_column}'
        table = pd.DataFrame({'ID': clean_sample_ids, 'sequence_length_aa': length_values})
    else:
        if not fasta_path.exists():
            raise FileNotFoundError(
                f'No recognized length column and FASTA not found: {fasta_path}. '
                'Upload the FASTA or edit its filename in DATASETS.'
            )
        fasta_lengths = read_fasta_lengths(fasta_path, id_mode)
        table = pd.DataFrame({'ID': clean_sample_ids}).merge(
            fasta_lengths, on='ID', how='left', validate='one_to_one'
        )
        source = f'FASTA: {fasta_path.name}'

    if table['sequence_length_aa'].isna().any():
        missing = table.loc[table['sequence_length_aa'].isna(), 'ID'].tolist()
        raise ValueError(f'Sequence lengths missing for {len(missing)} IDs: {missing[:15]}')
    if (table['sequence_length_aa'] <= 0).any():
        raise ValueError('All sequence lengths must be positive.')
    table['sequence_length_aa'] = table['sequence_length_aa'].astype(float)
    table['log10_sequence_length'] = np.log10(table['sequence_length_aa'])
    return table, source


In [ ]:
def select_descriptors(X, threshold=0.60):
    """Retain the maximum-variance descriptor from each correlation component."""
    corr = X.corr().abs()
    columns = list(X.columns)
    adjacency = {c: set() for c in columns}
    for i, left in enumerate(columns):
        for j in range(i + 1, len(columns)):
            right = columns[j]
            if corr.iloc[i, j] > threshold:
                adjacency[left].add(right)
                adjacency[right].add(left)

    visited, components = set(), []
    for column in columns:
        if column in visited:
            continue
        stack, component = [column], []
        visited.add(column)
        while stack:
            current = stack.pop()
            component.append(current)
            for neighbor in adjacency[current]:
                if neighbor not in visited:
                    visited.add(neighbor)
                    stack.append(neighbor)
        components.append(sorted(component))

    variances = X.var(axis=0)
    retained, records = [], []
    for cluster_id, component in enumerate(components, start=1):
        representative = variances.loc[component].idxmax()
        retained.append(representative)
        for feature in component:
            records.append({
                'feature': feature,
                'cluster_id': cluster_id,
                'cluster_size': len(component),
                'representative': representative,
                'status': 'retained' if feature == representative else 'excluded',
            })
    retained_set = set(retained)
    retained = [c for c in columns if c in retained_set]
    return retained, pd.DataFrame(records)


def pca_from_matrix(matrix, n_components=10):
    n_pc = min(n_components, matrix.shape[0], matrix.shape[1])
    if n_pc < 2:
        raise ValueError('At least two components are required.')
    pca = PCA(n_components=n_pc)
    scores = pca.fit_transform(matrix)
    return pca, scores


def benjamini_hochberg(p_values):
    """Return BH-adjusted P values in the original order."""
    p = np.asarray(p_values, dtype=float)
    n = len(p)
    order = np.argsort(p)
    ranked = p[order]
    adjusted_ranked = ranked * n / np.arange(1, n + 1)
    adjusted_ranked = np.minimum.accumulate(adjusted_ranked[::-1])[::-1]
    adjusted_ranked = np.clip(adjusted_ranked, 0, 1)
    adjusted = np.empty(n)
    adjusted[order] = adjusted_ranked
    return adjusted


In [ ]:
# Run the primary and length-adjusted PCA analyses.
all_results = {}
correlation_rows = []
score_tables = []
selection_tables = []
length_tables = []
comparison_rows = []

for dataset, cfg in DATASETS.items():
    if not cfg['csv'].exists():
        raise FileNotFoundError(f'Missing CSV for {dataset}: {cfg["csv"]}')

    df = pd.read_csv(cfg['csv'])
    id_column = df.columns[0]
    clean_sample_ids = clean_ids(df[id_column], cfg['id_mode'])
    feature_columns = [c for c in df.columns if str(c).endswith('_MolePct')]
    if not feature_columns:
        raise ValueError(f'No *_MolePct columns found in {cfg["csv"]}')

    X_raw = df[feature_columns].astype(float).copy()
    X_raw.index = clean_sample_ids
    if X_raw.isna().any().any():
        raise ValueError(f'Missing MolePct values in {dataset}')

    lengths, length_source = get_lengths(
        df, clean_sample_ids, cfg['fasta'], cfg['id_mode']
    )
    lengths.insert(0, 'dataset', dataset)
    lengths['length_source'] = length_source
    length_tables.append(lengths)

    retained, selection = select_descriptors(
        X_raw, threshold=CORRELATION_THRESHOLD
    )
    selection.insert(0, 'dataset', dataset)
    selection_tables.append(selection)

    # Primary PCA: StandardScaler applied to retained MolePct descriptors.
    X_selected = X_raw[retained]
    X_standardized = StandardScaler().fit_transform(X_selected)
    original_pca, original_scores = pca_from_matrix(
        X_standardized, n_components=N_COMPONENTS
    )

    # Correlate length with every available PC up to PCS_TO_TEST.
    log_length = lengths['log10_sequence_length'].to_numpy()
    n_test_pc = min(PCS_TO_TEST, original_scores.shape[1])
    for pc_index in range(n_test_pc):
        rho, p_value = spearmanr(log_length, original_scores[:, pc_index])
        correlation_rows.append({
            'dataset': dataset,
            'PC': f'PC{pc_index + 1}',
            'n': len(log_length),
            'spearman_rho': rho,
            'p_value': p_value,
        })

    # Residualize each standardized descriptor against log10(length).
    length_predictor = log_length.reshape(-1, 1)
    residual_matrix = np.empty_like(X_standardized, dtype=float)
    descriptor_length_rows = []
    for j, feature in enumerate(retained):
        model = LinearRegression().fit(length_predictor, X_standardized[:, j])
        fitted = model.predict(length_predictor)
        residual_matrix[:, j] = X_standardized[:, j] - fitted
        descriptor_rho, descriptor_p = spearmanr(
            log_length, X_standardized[:, j]
        )
        descriptor_length_rows.append({
            'dataset': dataset,
            'descriptor': feature,
            'linear_slope_per_log10_length': model.coef_[0],
            'linear_R2': model.score(length_predictor, X_standardized[:, j]),
            'spearman_rho': descriptor_rho,
            'spearman_p': descriptor_p,
        })

    # Rescale residuals so that the length-adjusted PCA treats descriptors equally.
    residual_scaled = StandardScaler().fit_transform(residual_matrix)
    adjusted_pca, adjusted_scores = pca_from_matrix(
        residual_scaled, n_components=N_COMPONENTS
    )

    distance_rho, distance_p = spearmanr(
        pdist(original_scores[:, :2]),
        pdist(adjusted_scores[:, :2]),
    )
    _, _, disparity = procrustes(
        original_scores[:, :2], adjusted_scores[:, :2]
    )

    comparison_rows.append({
        'dataset': dataset,
        'n_sequences': len(clean_sample_ids),
        'n_retained_descriptors': len(retained),
        'original_PC1_percent': original_pca.explained_variance_ratio_[0] * 100,
        'original_PC2_percent': original_pca.explained_variance_ratio_[1] * 100,
        'original_PC1_PC2_percent': original_pca.explained_variance_ratio_[:2].sum() * 100,
        'length_adjusted_PC1_percent': adjusted_pca.explained_variance_ratio_[0] * 100,
        'length_adjusted_PC2_percent': adjusted_pca.explained_variance_ratio_[1] * 100,
        'length_adjusted_PC1_PC2_percent': adjusted_pca.explained_variance_ratio_[:2].sum() * 100,
        'distance_spearman_rho_original_vs_adjusted': distance_rho,
        'distance_spearman_p_original_vs_adjusted': distance_p,
        'procrustes_disparity_original_vs_adjusted': disparity,
        'retained_descriptors': ';'.join(retained),
        'length_source': length_source,
    })

    score_table = pd.DataFrame({
        'dataset': dataset,
        'ID': clean_sample_ids,
        'sequence_length_aa': lengths['sequence_length_aa'],
        'log10_sequence_length': log_length,
        'original_PC1': original_scores[:, 0],
        'original_PC2': original_scores[:, 1],
        'length_adjusted_PC1': adjusted_scores[:, 0],
        'length_adjusted_PC2': adjusted_scores[:, 1],
    })
    score_tables.append(score_table)

    all_results[dataset] = {
        'lengths': lengths,
        'retained': retained,
        'original_pca': original_pca,
        'original_scores': original_scores,
        'adjusted_pca': adjusted_pca,
        'adjusted_scores': adjusted_scores,
        'descriptor_length_table': pd.DataFrame(descriptor_length_rows),
    }

correlations = pd.DataFrame(correlation_rows)
correlations['adjusted_p_value_BH'] = benjamini_hochberg(correlations['p_value'])
comparisons = pd.DataFrame(comparison_rows)
scores_combined = pd.concat(score_tables, ignore_index=True)
lengths_combined = pd.concat(length_tables, ignore_index=True)
selections_combined = pd.concat(selection_tables, ignore_index=True)
descriptor_length_combined = pd.concat(
    [r['descriptor_length_table'] for r in all_results.values()],
    ignore_index=True,
)

correlations.to_csv(OUTPUT_DIR / 'sequence_length_PC_correlations.csv', index=False)
comparisons.to_csv(OUTPUT_DIR / 'original_vs_length_adjusted_PCA_summary.csv', index=False)
scores_combined.to_csv(OUTPUT_DIR / 'PCA_scores_with_sequence_lengths.csv', index=False)
lengths_combined.to_csv(OUTPUT_DIR / 'sequence_lengths_used.csv', index=False)
selections_combined.to_csv(OUTPUT_DIR / 'descriptor_selection_r060.csv', index=False)
descriptor_length_combined.to_csv(
    OUTPUT_DIR / 'descriptor_sequence_length_relationships.csv', index=False
)

display(correlations.round(5))
display(comparisons.round(5))


In [ ]:
# Plot log10(sequence length) against PC1 and PC2.
fig, axes = plt.subplots(3, 2, figsize=(11, 13), constrained_layout=True)

for row, dataset in enumerate(DATASETS):
    result = all_results[dataset]
    log_length = result['lengths']['log10_sequence_length'].to_numpy()
    scores = result['original_scores']

    for col, pc_index in enumerate([0, 1]):
        ax = axes[row, col]
        y = scores[:, pc_index]
        ax.scatter(log_length, y, s=22, alpha=0.75, color='#4c78a8')
        slope, intercept = np.polyfit(log_length, y, 1)
        x_line = np.linspace(log_length.min(), log_length.max(), 100)
        ax.plot(x_line, slope * x_line + intercept, color='#d62728', linewidth=1.2)
        rho, p_value = spearmanr(log_length, y)
        ax.text(
            0.03, 0.97, f'Spearman rho={rho:.3f}\nP={p_value:.3g}',
            transform=ax.transAxes, va='top'
        )
        ax.set_xlabel('log10(sequence length, aa)')
        ax.set_ylabel(f'PC{pc_index + 1} score')
        ax.set_title(f'{dataset}: sequence length vs PC{pc_index + 1}')

fig.savefig(OUTPUT_DIR / 'sequence_length_vs_PC1_PC2.png', dpi=300, bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'sequence_length_vs_PC1_PC2.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# Compare the original and length-adjusted PC1-PC2 configurations.
fig, axes = plt.subplots(3, 2, figsize=(11, 13), constrained_layout=True)

for row, dataset in enumerate(DATASETS):
    result = all_results[dataset]
    log_length = result['lengths']['log10_sequence_length'].to_numpy()

    for col, (label, scores, pca) in enumerate([
        ('Original PCA', result['original_scores'], result['original_pca']),
        ('Length-adjusted PCA', result['adjusted_scores'], result['adjusted_pca']),
    ]):
        ax = axes[row, col]
        scatter = ax.scatter(
            scores[:, 0], scores[:, 1], c=log_length,
            cmap='viridis', s=24, alpha=0.8
        )
        ax.axhline(0, color='#bdbdbd', linewidth=0.7)
        ax.axvline(0, color='#bdbdbd', linewidth=0.7)
        ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
        ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
        ax.set_title(f'{dataset}: {label}')
        fig.colorbar(scatter, ax=ax, label='log10(sequence length, aa)')

fig.savefig(OUTPUT_DIR / 'original_vs_length_adjusted_PCA.png', dpi=300, bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'original_vs_length_adjusted_PCA.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# Package and download all results.
readme = OUTPUT_DIR / 'README.txt'
readme.write_text(
    'Sequence-length effect analysis for the amino-acid-composition PCA.\n'
    'Primary descriptor-correlation threshold: |r| > 0.60.\n'
    'PC-length associations use Spearman correlations.\n'
    'Adjusted P values use the Benjamini-Hochberg method across all tested dataset-PC pairs.\n'
    'Length-adjusted PCA uses residuals after linear regression of each retained standardized descriptor on log10 sequence length.\n'
    'High original-vs-adjusted distance correlation and low Procrustes disparity indicate limited influence of sequence length on the overall PC1-PC2 configuration.\n',
    encoding='utf-8',
)

zip_path = Path(shutil.make_archive(
    str(BASE_DIR / 'PCA_sequence_length_effects_outputs'),
    'zip', root_dir=str(OUTPUT_DIR)
))
print('Created:', zip_path)

from google.colab import files
files.download(str(zip_path))
